### Source Tables:
- _exponent._bronze_allscripts_tw_works_vw.dbo_item_result (observation header - 413M records)
- _exponent._bronze_allscripts_tw_works_vw.dbo_result (observation values/results - 427M records)
- _exponent._bronze_allscripts_tw_works_vw.dbo_item_finding (findings header - 283M records)
- _exponent._bronze_allscripts_tw_works_vw.dbo_finding (finding values - 298M records)

### To Do:
- Map QODE to OMOP observation_concept_id using domain_source_to_concept
- Map observation_type_concept_id (default: 44818701 = From EHR)
- Link observations to visits via ActivityHeaderID → order_activity_header → EncounterID
- Map unit_concept_id from UnitsDE/UnitsDET
- Handle both numeric results (NumericResult) and text results (AnswerDET)
- Link to provider_id once provider table is populated

### Notes:
- PERSON and VISIT_OCCURRENCE must run before OBSERVATION
- dbo_item_result.ID is the observation identifier
- dbo_item_result.CurrentID links to dbo_result.ID for actual values
- dbo_item_result.PatientID links to dbo_person.ID
- dbo_item_result.ActivityHeaderID links to order_activity_header (for visit context)
- QODE = observation type code
- Starting with dbo_item_result/dbo_result (can add findings later)

In [ ]:
%sql
-- Check how many observations we have (sample to avoid timeout)
SELECT 
    COUNT(*) as total_observations,
    COUNT(DISTINCT PatientID) as unique_patients
FROM (
    SELECT PatientID 
    FROM `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_item_result`
    WHERE PatientID IS NOT NULL 
    LIMIT 10000
)

# Transformation

In [ ]:
source = 'allscripts_tw'

In [ ]:
silver_observation_df = spark.sql(f'''
SELECT 
  source_to_person.person_id,
  COALESCE(obs_concept.omop_concept_id, 0) AS observation_concept_id,  -- Map QODE to OMOP concept
  CAST(COALESCE(r.ClinicalDTTM, ir.PerformedDTTM) AS DATE) AS observation_date,
  COALESCE(r.ClinicalDTTM, ir.PerformedDTTM) AS observation_datetime,
  44818701 AS observation_type_concept_id,  -- 44818701 = EHR
  r.NumericResult AS value_as_number,
  CASE 
    WHEN r.NumericResult IS NULL THEN COALESCE(r.AnswerDET, ir.DecodedValue)
    ELSE NULL 
  END AS value_as_string,
  NULL AS value_as_concept_id,  -- TODO: Map answer codes to concepts if available
  NULL AS qualifier_concept_id,
  0 AS unit_concept_id,  -- TODO: Map UnitsDE/UnitsDET to OMOP unit concepts
  NULL AS provider_id,  -- TODO: Map WhoDidItID once provider table is populated
  source_to_visit_occurrence.visit_occurrence_id,
  NULL AS visit_detail_id,
  CONCAT('{source}', ' | ', ir.ID) AS observation_source_value,
  0 AS observation_source_concept_id,
  r.UnitsDET AS unit_source_value,
  NULL AS qualifier_source_value,
  COALESCE(r.AnswerDET, ir.DecodedValue) AS value_source_value,
  NULL AS observation_event_id,
  NULL AS obs_event_field_concept_id,
  '{source}' AS source_system
FROM `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_item_result` ir
INNER JOIN _exponent.omop_mapping.source_to_person
  ON CONCAT('{source}', CHAR(31), 'dbo_person', CHAR(31), 'id', CHAR(31), CAST(ir.PatientID AS BIGINT)) = source_to_person.person_source_value
  AND source_to_person.active_flag = TRUE
LEFT JOIN `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_result` r
  ON ir.CurrentID = r.ID
LEFT JOIN `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_order_activity_header` oah
  ON ir.ActivityHeaderID = oah.ID
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence
  ON CONCAT('{source}', ' | ', oah.EncounterID) = source_to_visit_occurrence.visit_occurrence_source_value
  AND source_to_visit_occurrence.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept obs_concept
  ON obs_concept.source_id = ir.QODE
  AND obs_concept.domain_id = 'Observation'
  AND obs_concept.source_system = '{source}'
WHERE ir.ID IS NOT NULL
  AND ir.PatientID IS NOT NULL
''')

display(silver_observation_df)
silver_observation_df.createOrReplaceTempView("silver_observation")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.observation AS t
USING silver_observation AS s
ON t.observation_source_value = s.observation_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.observation_concept_id <=> s.observation_concept_id)
  OR NOT (t.observation_date <=> s.observation_date)
  OR NOT (t.observation_datetime <=> s.observation_datetime)
  OR NOT (t.observation_type_concept_id <=> s.observation_type_concept_id)
  OR NOT (t.value_as_number <=> s.value_as_number)
  OR NOT (t.value_as_string <=> s.value_as_string)
  OR NOT (t.value_as_concept_id <=> s.value_as_concept_id)
  OR NOT (t.qualifier_concept_id <=> s.qualifier_concept_id)
  OR NOT (t.unit_concept_id <=> s.unit_concept_id)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.visit_detail_id <=> s.visit_detail_id)
  OR NOT (t.observation_source_concept_id <=> s.observation_source_concept_id)
  OR NOT (t.unit_source_value <=> s.unit_source_value)
  OR NOT (t.qualifier_source_value <=> s.qualifier_source_value)
  OR NOT (t.value_source_value <=> s.value_source_value)
  OR NOT (t.observation_event_id <=> s.observation_event_id)
  OR NOT (t.obs_event_field_concept_id <=> s.obs_event_field_concept_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                      = s.person_id,
  t.observation_concept_id         = s.observation_concept_id,
  t.observation_date               = s.observation_date,
  t.observation_datetime           = s.observation_datetime,
  t.observation_type_concept_id    = s.observation_type_concept_id,
  t.value_as_number                = s.value_as_number,
  t.value_as_string                = s.value_as_string,
  t.value_as_concept_id            = s.value_as_concept_id,
  t.qualifier_concept_id           = s.qualifier_concept_id,
  t.unit_concept_id                = s.unit_concept_id,
  t.provider_id                    = s.provider_id,
  t.visit_occurrence_id            = s.visit_occurrence_id,
  t.visit_detail_id                = s.visit_detail_id,
  t.observation_source_concept_id  = s.observation_source_concept_id,
  t.unit_source_value              = s.unit_source_value,
  t.qualifier_source_value         = s.qualifier_source_value,
  t.value_source_value             = s.value_source_value,
  t.observation_event_id           = s.observation_event_id,
  t.obs_event_field_concept_id     = s.obs_event_field_concept_id,
  t.source_system                  = s.source_system

WHEN NOT MATCHED THEN INSERT (
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id,
  source_system
)
VALUES (
  s.person_id,
  s.observation_concept_id,
  s.observation_date,
  s.observation_datetime,
  s.observation_type_concept_id,
  s.value_as_number,
  s.value_as_string,
  s.value_as_concept_id,
  s.qualifier_concept_id,
  s.unit_concept_id,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.observation_source_value,
  s.observation_source_concept_id,
  s.unit_source_value,
  s.qualifier_source_value,
  s.value_source_value,
  s.observation_event_id,
  s.obs_event_field_concept_id,
  s.source_system
);

In [ ]:
%sql
SELECT * FROM _exponent.omop_silver.observation
LIMIT 10

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_observation (
    source_system,
    observation_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.observation_source_value,
    s.person_id,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 
        source_system, 
        observation_source_value,
        person_id
    FROM _exponent.omop_silver.observation
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation x
  ON s.observation_source_value = x.observation_source_value;

In [ ]:
%sql
SELECT * FROM _exponent.omop_mapping.source_to_observation
LIMIT 20

In [ ]:
%sql
MERGE INTO _exponent.omop.observation AS gold_obs
USING (
  SELECT
    source_to_observation.observation_id,
    s.person_id,
    s.observation_concept_id,
    s.observation_date,
    s.observation_datetime,
    s.observation_type_concept_id,
    s.value_as_number,
    s.value_as_string,
    s.value_as_concept_id,
    s.qualifier_concept_id,
    s.unit_concept_id,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.observation_source_value,
    s.observation_source_concept_id,
    s.unit_source_value,
    s.qualifier_source_value,
    s.value_source_value,
    s.observation_event_id,
    s.obs_event_field_concept_id
  FROM _exponent.omop_silver.observation s
  JOIN _exponent.omop_mapping.source_to_observation
    ON source_to_observation.observation_source_value = s.observation_source_value
   AND source_to_observation.active_flag = TRUE
) AS src
ON gold_obs.observation_id = src.observation_id

WHEN MATCHED THEN UPDATE SET
  gold_obs.person_id                     = src.person_id,
  gold_obs.observation_concept_id        = src.observation_concept_id,
  gold_obs.observation_date              = src.observation_date,
  gold_obs.observation_datetime          = src.observation_datetime,
  gold_obs.observation_type_concept_id   = src.observation_type_concept_id,
  gold_obs.value_as_number               = src.value_as_number,
  gold_obs.value_as_string               = src.value_as_string,
  gold_obs.value_as_concept_id           = src.value_as_concept_id,
  gold_obs.qualifier_concept_id          = src.qualifier_concept_id,
  gold_obs.unit_concept_id               = src.unit_concept_id,
  gold_obs.provider_id                   = src.provider_id,
  gold_obs.visit_occurrence_id           = src.visit_occurrence_id,
  gold_obs.visit_detail_id               = src.visit_detail_id,
  gold_obs.observation_source_value      = src.observation_source_value,
  gold_obs.observation_source_concept_id = src.observation_source_concept_id,
  gold_obs.unit_source_value             = src.unit_source_value,
  gold_obs.qualifier_source_value        = src.qualifier_source_value,
  gold_obs.value_source_value            = src.value_source_value,
  gold_obs.observation_event_id          = src.observation_event_id,
  gold_obs.obs_event_field_concept_id    = src.obs_event_field_concept_id

WHEN NOT MATCHED THEN INSERT (
  observation_id,
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id
)
VALUES (
  src.observation_id,
  src.person_id,
  src.observation_concept_id,
  src.observation_date,
  src.observation_datetime,
  src.observation_type_concept_id,
  src.value_as_number,
  src.value_as_string,
  src.value_as_concept_id,
  src.qualifier_concept_id,
  src.unit_concept_id,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.observation_source_value,
  src.observation_source_concept_id,
  src.unit_source_value,
  src.qualifier_source_value,
  src.value_source_value,
  src.observation_event_id,
  src.obs_event_field_concept_id
);

In [ ]:
%sql
SELECT * FROM _exponent.omop.observation
LIMIT 20